# 🧠 03 — Model Training

Train all three transfer learning models:
- VGG16
- ResNet50
- InceptionV3

Each model uses a two-phase training strategy:
1. **Phase 1:** Feature extraction (frozen base)
2. **Phase 2:** Fine-tuning (unfrozen top layers)

In [ ]:
import sys
sys.path.append('..')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Reduce TF verbosity

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

import config
from src.utils.helpers import set_seed, get_device_info
from src.models.model_factory import ModelFactory
from src.training.trainer import Trainer

%matplotlib inline
plt.style.use('dark_background')

set_seed(42)
device_info = get_device_info()
print(f'TensorFlow: {device_info["tf_version"]}')
print(f'GPUs available: {device_info["gpus"]}')
print(f'GPU names: {device_info["gpu_names"]}')

## 1. Model Architecture Overview

In [ ]:
print('Available models:', ModelFactory.list_models())
print()

for name in ModelFactory.list_models():
    model = ModelFactory.create(name, build=True)
    total = model.model.count_params()
    trainable = sum(np.prod(w.shape) for w in model.model.trainable_weights)
    print(f'{name.upper():15s} | Input: {model.input_shape} | '
          f'Total: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M')
    del model

## 2. Train ResNet50

Start with ResNet50 as the default model.

In [ ]:
trainer_resnet = Trainer(
    model_name='resnet50',
    batch_size=config.BATCH_SIZE,
    epochs_phase1=config.EPOCHS_PHASE1,
    epochs_phase2=config.EPOCHS_PHASE2,
)

results_resnet = trainer_resnet.train()
trainer_resnet.save_results(results_resnet)

## 3. Train VGG16

In [ ]:
trainer_vgg = Trainer(
    model_name='vgg16',
    batch_size=config.BATCH_SIZE,
    epochs_phase1=config.EPOCHS_PHASE1,
    epochs_phase2=config.EPOCHS_PHASE2,
)

results_vgg = trainer_vgg.train()
trainer_vgg.save_results(results_vgg)

## 4. Train InceptionV3

In [ ]:
trainer_inception = Trainer(
    model_name='inceptionv3',
    batch_size=config.BATCH_SIZE,
    epochs_phase1=config.EPOCHS_PHASE1,
    epochs_phase2=config.EPOCHS_PHASE2,
)

results_inception = trainer_inception.train()
trainer_inception.save_results(results_inception)

## 5. Training Curves Comparison

In [ ]:
from src.evaluation.visualizations import plot_training_history

all_results = {
    'resnet50': results_resnet,
    'vgg16': results_vgg,
    'inceptionv3': results_inception,
}

for name, res in all_results.items():
    # Combine phase1 + phase2 history for plotting
    combined = {}
    for key in res['phase1']:
        combined[key] = res['phase1'][key] + res['phase2'].get(key, [])
    
    path = plot_training_history(combined, model_name=name)
    print(f'Saved: {path}')
    
    # Display inline
    img = plt.imread(path)
    plt.figure(figsize=(14, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

---
**Next:** Proceed to `04_model_evaluation.ipynb` for detailed evaluation.